# Etapa 1 — Aprendizado Simbólico: árvore de decisão

**Pergunta prática:** dado o perfil e os sintomas de um paciente com dengue notificado, ele será hospitalizado?

**Dados:** Sinan/Dengue 2026 (`DENGBR26.csv`), Portal de Dados Abertos do SUS.

Rode as células em ordem (Ambiente de execução → Executar tudo).

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier

SEED = 42
CAMINHO_CSV = "/content/DENGBR26.csv"

## 1. Funções (ID3 do zero)

In [ ]:
def entropia(y):
    y = list(y)
    if len(y) == 0:
        return 0.0
    p = np.array(list(Counter(y).values())) / len(y)
    return max(0.0, float(-np.sum(p * np.log2(p))))


def ganho(df, attr, alvo="alvo"):
    h_pai = entropia(df[alvo])
    h_filhos = sum(len(sub) / len(df) * entropia(sub[alvo])
                   for _, sub in df.groupby(attr))
    return h_pai - h_filhos


def arvore(df, atributos, alvo="alvo", max_prof=None, prof=0):
    """ID3 com entropia. max_prof=None -> sem poda."""
    maj = int(df[alvo].mode()[0])
    if df[alvo].nunique() == 1:
        return int(df[alvo].iloc[0])
    if not atributos or (max_prof is not None and prof >= max_prof):
        return maj
    A = max(atributos, key=lambda a: ganho(df, a, alvo))
    return {
        "atributo": A,
        "maj": maj,
        "filhos": {v: arvore(sub, [a for a in atributos if a != A],
                             alvo, max_prof, prof + 1)
                   for v, sub in df.groupby(A)},
    }


def prever(no, linha):
    while isinstance(no, dict):
        v = linha[no["atributo"]]
        if v not in no["filhos"]:      # valor nunca visto neste ramo
            return no["maj"]
        no = no["filhos"][v]
    return no


def acuracia(no, df, alvo="alvo"):
    linhas = df.to_dict("records")
    return sum(prever(no, l) == l[alvo] for l in linhas) / len(linhas)


def faixa_etaria(x):
    # NU_IDADE_N: 1o dígito = unidade (4 = anos), demais = valor. Ex.: 4048 = 48 anos
    if pd.isna(x):
        return "ignorado"
    x = int(x)
    unidade, valor = x // 1000, x % 1000
    anos = valor if unidade == 4 else 0     # horas, dias e meses = menos de 1 ano
    if anos < 12:
        return "crianca"
    elif anos < 60:
        return "adulto"
    return "idoso"


def regiao(x):
    # SG_UF_NOT: código IBGE do estado; o 1o dígito é a região
    if pd.isna(x):
        return "ignorado"
    d = str(int(float(x)))[0]
    return {"1": "norte", "2": "nordeste", "3": "sudeste",
            "4": "sul", "5": "centro-oeste"}.get(d, "ignorado")


def preparar(df, atributos, seed=SEED):
    """Atributos como texto (nulo -> 'ignorado') + undersampling 50/50."""
    d = df[atributos + ["alvo"]].copy()
    for a in atributos:
        d[a] = d[a].fillna("ignorado").astype(str)
    pos = d[d["alvo"] == 1]
    neg = d[d["alvo"] == 0].sample(n=len(pos), random_state=seed)
    return pd.concat([pos, neg]).sample(frac=1, random_state=seed).reset_index(drop=True)


def dummies(tr, te, attrs):
    """scikit-learn precisa de números: atributos viram colunas 0/1."""
    X_tr = pd.get_dummies(tr[attrs])
    X_te = pd.get_dummies(te[attrs]).reindex(columns=X_tr.columns, fill_value=0)
    return X_tr, X_te


def cv_id3(dados, atributos, max_prof=None, k=5):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=SEED)
    dados = dados.reset_index(drop=True)
    accs = []
    for tr, va in skf.split(dados, dados["alvo"]):
        m = arvore(dados.iloc[tr], atributos, max_prof=max_prof)
        accs.append(acuracia(m, dados.iloc[va]))
    return accs


def relatorio_niveis(df, atributos, niveis=3, rotulo="raiz", nivel=1):
    """Imprime n, entropia e ganhos de cada nó até o nível indicado."""
    n = len(df)
    c0, c1 = int((df["alvo"] == 0).sum()), int((df["alvo"] == 1).sum())
    H = entropia(df["alvo"])
    ind = "    " * (nivel - 1)
    print(f"{ind}[nivel {nivel}] {rotulo} | n={n} (nao={c0}, sim={c1}) | H={H:.4f}")
    if H == 0 or not atributos:
        print(f"{ind}   (folha)")
        return
    gs = sorted(((a, ganho(df, a)) for a in atributos), key=lambda t: -t[1])
    print(f"{ind}   ganhos: " + " | ".join(f"{a}={g:.5f}" for a, g in gs))
    melhor = gs[0][0]
    print(f"{ind}   -> divide por: {melhor}")
    if nivel < niveis:
        resto = [a for a in atributos if a != melhor]
        for v, sub in df.groupby(melhor):
            relatorio_niveis(sub, resto, niveis, f"{melhor}={v}", nivel + 1)

## 2. Carga dos dados

In [ ]:
COLUNAS = ["HOSPITALIZ", "NU_IDADE_N", "SG_UF_NOT", "CS_SEXO", "VOMITO",
           "PETEQUIA_N", "DIABETES", "HIPERTENSA", "CEFALEIA", "LEUCOPENIA",
           "MIALGIA", "DOR_RETRO", "ARTRALGIA", "LACO", "EXANTEMA",
           "NAUSEA", "DOR_COSTAS"]

df = pd.read_csv(CAMINHO_CSV, sep=",", encoding="latin-1",
                 usecols=COLUNAS, low_memory=False)
print(df.shape)
print(df["HOSPITALIZ"].value_counts(dropna=False))
d = df[df["HOSPITALIZ"].isin([1, 2])]

(470451, 17)
HOSPITALIZ
2.0    317075
NaN    113632
1.0     29690
9.0     10053
0.0         1
Name: count, dtype: int64


## 3. Preparação: alvo, atributos derivados, balanceamento e divisão treino/teste

Alvo: `HOSPITALIZ` (1 = hospitalizou, 2 = não). Linhas sem desfecho conhecido (nulo, 9 ou 0) são descartadas.
Como só ~8,5% são hospitalizações, usamos *undersampling* para ficar 50/50 (linha de base = 50%).

In [ ]:
n_total = len(df)
df = df[df["HOSPITALIZ"].isin([1, 2])].copy()
df["alvo"] = (df["HOSPITALIZ"] == 1).astype(int)
df["faixa_etaria"] = df["NU_IDADE_N"].apply(faixa_etaria)
df["regiao"] = df["SG_UF_NOT"].apply(regiao)

print(f"registros no arquivo: {n_total}")
print(f"usados (HOSPITALIZ = 1 ou 2): {len(df)} | descartados: {n_total - len(df)}")
print("\nproporção do alvo antes de balancear:")
print(df["alvo"].value_counts(normalize=True).round(4))
print("\nregiões:");       print(df["regiao"].value_counts())
print("\nfaixas etárias:"); print(df["faixa_etaria"].value_counts())

CANDIDATOS = ["regiao", "SG_UF_NOT", "faixa_etaria", "CS_SEXO", "VOMITO",
              "PETEQUIA_N", "DIABETES", "HIPERTENSA", "CEFALEIA", "LEUCOPENIA",
              "MIALGIA", "DOR_RETRO", "ARTRALGIA", "LACO", "EXANTEMA",
              "NAUSEA", "DOR_COSTAS"]

dados_full = preparar(df, CANDIDATOS)
treino_full, teste_full = train_test_split(
    dados_full, test_size=0.2, random_state=SEED, stratify=dados_full["alvo"])
treino_full = treino_full.reset_index(drop=True)
teste_full = teste_full.reset_index(drop=True)

print("\nbalanceado:", dados_full.shape, "| proporção:", dados_full["alvo"].mean())
print("treino:", treino_full.shape, "| teste:", teste_full.shape)

registros no arquivo: 470451
usados (HOSPITALIZ = 1 ou 2): 346765 | descartados: 123686

proporção do alvo antes de balancear:
alvo
0    0.9144
1    0.0856
Name: proportion, dtype: float64

regiões:
regiao
sudeste         126527
centro-oeste    109840
nordeste         76589
norte            22337
sul              11472
Name: count, dtype: int64

faixas etárias:
faixa_etaria
adulto     261993
crianca     50747
idoso       34025
Name: count, dtype: int64

balanceado: (59380, 18) | proporção: 0.5
treino: (47504, 18) | teste: (11876, 18)


## 4. Seleção de atributos

Ganho de informação de cada candidato, medido **só no treino**. Colunas excluídas por vazamento do alvo:
`DT_INTERNA`, `CLASSI_FIN`, `CRITERIO`, `EVOLUCAO`, `ALRM_*`, `GRAV_*` e similares.

In [ ]:
ganhos_cand = pd.Series({c: ganho(treino_full, c) for c in CANDIDATOS}).sort_values(ascending=False)
print(ganhos_cand.round(5).to_string())

# escolha final (6 atributos interpretáveis)
ATRIBUTOS = ["regiao", "LEUCOPENIA", "faixa_etaria", "VOMITO", "CEFALEIA", "HIPERTENSA"]

treino = treino_full[ATRIBUTOS + ["alvo"]].copy()
teste = teste_full[ATRIBUTOS + ["alvo"]].copy()
print("\nEntropia da raiz:", entropia(treino["alvo"]))

SG_UF_NOT       0.06640
regiao          0.04479
LEUCOPENIA      0.02936
faixa_etaria    0.01732
VOMITO          0.01510
CEFALEIA        0.01415
HIPERTENSA      0.00874
PETEQUIA_N      0.00859
DIABETES        0.00731
MIALGIA         0.00435
NAUSEA          0.00420
DOR_RETRO       0.00355
EXANTEMA        0.00213
LACO            0.00122
DOR_COSTAS      0.00066
CS_SEXO         0.00004
ARTRALGIA       0.00002

Entropia da raiz: 1.0


## 5. Evolução das versões de atributos (ID3 sem poda)

In [ ]:
VERSOES = {
    "v1 - atributos iniciais": ["faixa_etaria", "CS_SEXO", "VOMITO", "PETEQUIA_N", "DIABETES", "HIPERTENSA"],
    "v2 - com estado (SG_UF_NOT)": ["SG_UF_NOT", "LEUCOPENIA", "faixa_etaria", "VOMITO", "CEFALEIA", "HIPERTENSA"],
    "v3 - com regiao (FINAL)": ATRIBUTOS,
}

res = []
for nome, attrs in VERSOES.items():
    tr, te = treino_full[attrs + ["alvo"]], teste_full[attrs + ["alvo"]]
    m = arvore(tr, attrs)
    res.append({"versao": nome, "treino": acuracia(m, tr), "teste": acuracia(m, te)})
print(pd.DataFrame(res).round(4).to_string(index=False))

                     versao  treino  teste
    v1 - atributos iniciais  0.6052 0.6056
v2 - com estado (SG_UF_NOT)  0.6897 0.6865
    v3 - com regiao (FINAL)  0.6670 0.6632


## 6. Entropia e ganho — níveis 1 a 3 (versão final)

**Nível 1 (raiz):** tabela completa para a conta à mão.

In [ ]:
raiz = max(ATRIBUTOS, key=lambda a: ganho(treino, a))
print("atributo da raiz:", raiz, "\n")

tab = pd.crosstab(treino[raiz], treino["alvo"]).rename(columns={0: "nao", 1: "sim"})
tab["total"] = tab["nao"] + tab["sim"]
tab["p_sim"] = tab["sim"] / tab["total"]
tab["entropia"] = [entropia(treino.loc[treino[raiz] == v, "alvo"]) for v in tab.index]
tab["peso"] = tab["total"] / len(treino)

H_pai = entropia(treino["alvo"])
H_filhos = (tab["peso"] * tab["entropia"]).sum()

print(tab.round(5).to_string())
print(f"\nn (treino) = {len(treino)}")
print(f"H(pai) = {H_pai:.5f}")
print(f"H(filhos) = soma(peso x entropia) = {H_filhos:.5f}")
print(f"Ganho({raiz}) = H(pai) - H(filhos) = {H_pai - H_filhos:.5f}")
print(f"conferencia com ganho(): {ganho(treino, raiz):.5f}")

atributo da raiz: regiao 

alvo           nao   sim  total    p_sim  entropia     peso
regiao                                                     
centro-oeste  7623  5390  13013  0.41420   0.97865  0.27393
nordeste      4848  9544  14392  0.66315   0.92178  0.30296
norte         1471  2383   3854  0.61832   0.95922  0.08113
sudeste       9025  5645  14670  0.38480   0.96136  0.30882
sul            785   790   1575  0.50159   0.99999  0.03316

n (treino) = 47504
H(pai) = 1.00000
H(filhos) = soma(peso x entropia) = 0.95521
Ganho(regiao) = H(pai) - H(filhos) = 0.04479
conferencia com ganho(): 0.04479


**Níveis 1 a 3:** ganhos de todos os atributos em cada nó e o atributo escolhido.

In [ ]:
relatorio_niveis(treino, ATRIBUTOS, niveis=3)

[nivel 1] raiz | n=47504 (nao=23752, sim=23752) | H=1.0000
   ganhos: regiao=0.04479 | LEUCOPENIA=0.02936 | faixa_etaria=0.01732 | VOMITO=0.01510 | CEFALEIA=0.01415 | HIPERTENSA=0.00874
   -> divide por: regiao
    [nivel 2] regiao=centro-oeste | n=13013 (nao=7623, sim=5390) | H=0.9787
       ganhos: LEUCOPENIA=0.03509 | faixa_etaria=0.02190 | HIPERTENSA=0.01922 | CEFALEIA=0.01591 | VOMITO=0.01291
       -> divide por: LEUCOPENIA
        [nivel 3] LEUCOPENIA=1.0 | n=1185 (nao=288, sim=897) | H=0.8001
           ganhos: VOMITO=0.01556 | faixa_etaria=0.00767 | HIPERTENSA=0.00354 | CEFALEIA=0.00010
           -> divide por: VOMITO
        [nivel 3] LEUCOPENIA=2.0 | n=11828 (nao=7335, sim=4493) | H=0.9579
           ganhos: faixa_etaria=0.02288 | HIPERTENSA=0.01983 | CEFALEIA=0.01908 | VOMITO=0.01010
           -> divide por: faixa_etaria
    [nivel 2] regiao=nordeste | n=14392 (nao=4848, sim=9544) | H=0.9218
       ganhos: VOMITO=0.02319 | LEUCOPENIA=0.02193 | faixa_etaria=0.00896 | CEFAL

## 7. ID3 do zero vs scikit-learn (sem poda)

In [ ]:
modelo_id3 = arvore(treino, ATRIBUTOS)
X_tr, X_te = dummies(treino, teste, ATRIBUTOS)
clf = DecisionTreeClassifier(criterion="entropy", random_state=SEED).fit(X_tr, treino["alvo"])

print(f"ID3 treino:      {acuracia(modelo_id3, treino):.4f}")
print(f"ID3 teste:       {acuracia(modelo_id3, teste):.4f}")
print(f"sklearn treino:  {clf.score(X_tr, treino['alvo']):.4f}")
print(f"sklearn teste:   {clf.score(X_te, teste['alvo']):.4f}")

ID3 treino:      0.6670
ID3 teste:       0.6632
sklearn treino:  0.6670
sklearn teste:   0.6632


## 8. Profundidade ótima e validação cruzada 5-fold

Curva do scikit-learn (`max_depth` conta divisões binárias sobre colunas 0/1, não é idêntica à profundidade do ID3).

In [ ]:
for d in range(1, 9):
    c = DecisionTreeClassifier(criterion="entropy", max_depth=d, random_state=SEED).fit(X_tr, treino["alvo"])
    print(f"sklearn profundidade {d}: treino={c.score(X_tr, treino['alvo']):.4f}  teste={c.score(X_te, teste['alvo']):.4f}")

sklearn profundidade 1: treino=0.5989  teste=0.5937
sklearn profundidade 2: treino=0.6300  teste=0.6299
sklearn profundidade 3: treino=0.6439  teste=0.6442
sklearn profundidade 4: treino=0.6503  teste=0.6525
sklearn profundidade 5: treino=0.6571  teste=0.6519
sklearn profundidade 6: treino=0.6626  teste=0.6589
sklearn profundidade 7: treino=0.6662  teste=0.6628
sklearn profundidade 8: treino=0.6666  teste=0.6648


Profundidade do **ID3**, escolhida por 5-fold **dentro do treino** (sem olhar o teste). A profundidade ótima é a menor cuja média está a até 0,002 da melhor.

In [ ]:
medias = {}
for p in range(1, len(ATRIBUTOS) + 1):
    medias[p] = np.mean(cv_id3(treino, ATRIBUTOS, max_prof=p))
    print(f"ID3 profundidade {p}: media 5-fold (no treino) = {medias[p]:.4f}")

melhor_media = max(medias.values())
p_otima = min(p for p, m in medias.items() if m >= melhor_media - 0.002)
print("\nprofundidade otima (ID3):", p_otima)

ID3 profundidade 1: media 5-fold (no treino) = 0.6170
ID3 profundidade 2: media 5-fold (no treino) = 0.6427
ID3 profundidade 3: media 5-fold (no treino) = 0.6572
ID3 profundidade 4: media 5-fold (no treino) = 0.6589
ID3 profundidade 5: media 5-fold (no treino) = 0.6627
ID3 profundidade 6: media 5-fold (no treino) = 0.6642

profundidade otima (ID3): 5


Resultados finais: ID3 e scikit-learn, sem poda e com a profundidade ótima. A validação 5-fold usa o conjunto balanceado inteiro (treino + teste).

In [ ]:
modelo_full = arvore(treino, ATRIBUTOS)
modelo_poda = arvore(treino, ATRIBUTOS, max_prof=p_otima)

dados_cv = pd.concat([treino, teste]).reset_index(drop=True)
cv_full = cv_id3(dados_cv, ATRIBUTOS)
cv_poda = cv_id3(dados_cv, ATRIBUTOS, max_prof=p_otima)

X_all = pd.get_dummies(dados_cv[ATRIBUTOS])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
clf_poda = DecisionTreeClassifier(criterion="entropy", max_depth=p_otima, random_state=SEED).fit(X_tr, treino["alvo"])
cv_sk_full = cross_val_score(DecisionTreeClassifier(criterion="entropy", random_state=SEED), X_all, dados_cv["alvo"], cv=skf)
cv_sk_poda = cross_val_score(DecisionTreeClassifier(criterion="entropy", max_depth=p_otima, random_state=SEED), X_all, dados_cv["alvo"], cv=skf)

resumo = pd.DataFrame([
    {"modelo": "ID3 sem poda", "treino": acuracia(modelo_full, treino), "teste": acuracia(modelo_full, teste), "5-fold (media)": np.mean(cv_full)},
    {"modelo": f"ID3 prof={p_otima}", "treino": acuracia(modelo_poda, treino), "teste": acuracia(modelo_poda, teste), "5-fold (media)": np.mean(cv_poda)},
    {"modelo": "sklearn sem poda", "treino": clf.score(X_tr, treino["alvo"]), "teste": clf.score(X_te, teste["alvo"]), "5-fold (media)": cv_sk_full.mean()},
    {"modelo": f"sklearn max_depth={p_otima}", "treino": clf_poda.score(X_tr, treino["alvo"]), "teste": clf_poda.score(X_te, teste["alvo"]), "5-fold (media)": cv_sk_poda.mean()},
])
print(resumo.round(4).to_string(index=False))
print("\n5-fold ID3 sem poda:", [round(a, 4) for a in cv_full])
print("5-fold ID3 prof", p_otima, ":", [round(a, 4) for a in cv_poda])

             modelo  treino  teste  5-fold (media)
       ID3 sem poda  0.6670 0.6632          0.6627
         ID3 prof=5  0.6653 0.6639          0.6619
   sklearn sem poda  0.6670 0.6632          0.6627
sklearn max_depth=5  0.6571 0.6519          0.6560

5-fold ID3 sem poda: [0.6651, 0.6676, 0.6564, 0.6592, 0.6653]
5-fold ID3 prof 5 : [0.6604, 0.6655, 0.6593, 0.6586, 0.6658]


## 9. Salvar os conjuntos (para reuso na Etapa 2 e no repositório)

In [ ]:
treino.to_csv("treino_etapa1.csv", index=False)
teste.to_csv("teste_etapa1.csv", index=False)
print("salvos: treino_etapa1.csv, teste_etapa1.csv")

salvos: treino_etapa1.csv, teste_etapa1.csv


## Monotonicidade

Na lógica clássica, a dedução é monotônica: se uma conclusão decorre de um conjunto de premissas, ela continua valendo quando novas premissas são acrescentadas. O aprendizado indutivo não tem essa propriedade. A árvore de decisão generaliza a partir de uma amostra finita, e suas conclusões (qual atributo é o mais informativo, qual classe prever em cada folha) são hipóteses que dependem dos exemplos vistos. Um exemplo novo pode, portanto, invalidar uma conclusão anterior.

Isso ocorreu neste trabalho. Em uma execução anterior, o conjunto de treino tinha 13.689 exemplos (cerca de 30% do total) e o ID3 escolheu `LEUCOPENIA` como raiz, com ganho de informação 0,03517, por margem mínima sobre a região (0,03486). A conclusão "a leucopenia é o atributo mais informativo para prever a hospitalização" parecia sustentada pelos dados. Ao incorporar os demais exemplos (47.504 no treino), os ganhos mudaram e a conclusão deixou de valer:

| Conjunto de treino | Ganho de `regiao` | Ganho de `LEUCOPENIA` | Raiz escolhida |
|---|---|---|---|
| 13.689 exemplos | 0,03486 | 0,03517 | `LEUCOPENIA` |
| 47.504 exemplos | 0,04479 | 0,02936 | `regiao` |

Como todos os nós abaixo da raiz dependem dela, a árvore inteira mudou de forma, e não apenas um atributo.

Dois pontos explicam a fragilidade. Primeiro, o ganho é calculado sobre frequências observadas; quando os dois melhores atributos diferem por apenas 0,0003, poucos exemplos novos bastam para inverter a escolha. Segundo, a amostra parcial subrepresentava algumas regiões (o Sul tinha 5 casos no treino, contra 1.575 no conjunto completo), o que distorcia as estimativas.

Duas consequências práticas. Primeiro, a árvore final é provisória: os dados do Sinan são atualizados continuamente, e novas notificações podem alterar a estrutura. Segundo, a validação cruzada 5-fold estima o desempenho em dados não vistos (média de 66,3%, entre 65,6% e 66,8% nos folds), mas não garante que as conclusões se mantenham. Tratamos a árvore como o melhor modelo para os dados disponíveis em um dado momento, e aceitamos a possibilidade de revisão como característica do raciocínio indutivo, ao contrário da dedução da lógica clássica